# Practical P18: Token Limit Handling & Response Structure
**Learning Outcome**: Manage token limits intelligently in any AI API integration.

## Part 1: Token Counting with `tiktoken` or a String Tokenizer
LLMs charge by the token. We can use Python libraries to calculate token lengths before making API calls. This prevents hitting limits and helps manage pricing budgets.


In [ ]:
# Since students might not have tiktoken preinstalled, we will show a mock tokenizer function
# and also import tiktoken if available.
try:
    import tiktoken
    encoding = tiktoken.get_encoding('cl100k_base')
    def count_tokens(text):
        return len(encoding.encode(text))
    print('Using tiktoken library for counting.')
except ImportError:
    # Simple word-based estimation fallback
    def count_tokens(text):
        return int(len(text.split()) * 1.3)
    print('Using fallback word-estimation tokenizer.')

sample_prompt = 'Hello world! Today we are learning how to preprocess token values for Large Language Models.'
print(f"Token Count of '{sample_prompt}': {count_tokens(sample_prompt)} tokens")


## Part 2: Truncating Input Text to Fit Budget
If text exceeds our budget limits (e.g. 50 tokens max), we truncate it.


In [ ]:
def truncate_to_token_limit(text, max_tokens=15):
    words = text.split()
    # Keep appending words until we hit token limits
    truncated_words = []
    current_tokens = 0
    for w in words:
        est_token = count_tokens(w + ' ')
        if current_tokens + est_token > max_tokens:
            break
        truncated_words.append(w)
        current_tokens += est_token
    return ' '.join(truncated_words)

long_text = 'Large Language Models are deep learning algorithms that can recognize, summarize, translate, predict and generate content using very large datasets.'
print('Original:', long_text)
print('Truncated:', truncate_to_token_limit(long_text, max_tokens=10))


## Hands-On Exercise
**Task**: Create a validator function `process_prompt_with_tokens(prompt, max_budget=20)` that:
1. Checks token count of the prompt.
2. If prompt token length > `max_budget`, truncate the prompt and log warning details.
3. Return the cleaned/truncated prompt ready to send to an API.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('TokenValidator')

def process_prompt_with_tokens(prompt, max_budget=20):
    # TODO: Implement truncation validation
    tokens = count_tokens(prompt)
    if tokens > max_budget:
        logger.warning(f'Prompt token count ({tokens}) exceeds budget ({max_budget}). Truncating prompt!')
        prompt = truncate_to_token_limit(prompt, max_budget)
    else:
        logger.info(f'Prompt size is safe ({tokens} tokens).')
    return prompt

p1 = 'Short prompt.'
p2 = 'This is an exceptionally long prompt intended to test whether our validation logic is capable of truncating strings correctly.'

print('Processed P1:', process_prompt_with_tokens(p1))
print('Processed P2:', process_prompt_with_tokens(p2))
